# Forecaster tracker SDK — minimal example

Run `make install-sdk` once, then this notebook can be kernel-launched from anywhere under the repo.

In [ ]:
import tracker

with tracker.run(
    experiment="rf_crosssectional_us",
    params={
        "model": "random_forest",
        "top_k": 15,
        "window_months": 12,
        "min_train_rows": 150,
        "features": [
            "rsi", "garman_klass_volatility", "close_over_open", "atr", "macd",
            "bb_low", "bb_mid", "bb_high",
            "return_1m", "return_2m", "return_3m", "return_6m", "return_9m", "return_12m",
            "beta_mkt_rf", "beta_smb", "beta_hml", "beta_rmw", "beta_cma",
        ],
    },
    tags=["example", "baseline"],
    name="example_run",
) as r:
    # ... your training code here ...
    r.log_metric("oos_sharpe", 1.21)
    r.log_metric("oos_mae", 0.021)
    r.log_metric("train_r2", 0.18)
    r.log_metric("p_two_sided", 0.032)
    r.log_importance({"rsi": 0.12, "return_12m": 0.18, "macd": 0.08})
    # r.log_artifact("equity_curve", "data/artifacts/.../curve.png", kind="plot")
    print("run_id:", r.run_id)

In [ ]:
# Read the run back via raw sqlite3 (no Django needed from notebook).
import sqlite3, json
from tracker import resolve_warehouse_path

con = sqlite3.connect(resolve_warehouse_path())
con.row_factory = sqlite3.Row
for row in con.execute("""
    SELECT r.run_id, r.status, r.started_at, e.name AS exp
    FROM run r JOIN experiment e ON e.experiment_id = r.experiment_id
    ORDER BY r.started_at DESC LIMIT 5
"""):
    print(dict(row))